In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

from catboost import CatBoostClassifier, Pool

from sklearn.metrics import accuracy_score, classification_report

In [2]:
df = pd.read_csv("/home/jupyter/project/course work/predictive-models-for-football/data/selected_leagues_one_line.csv")

In [3]:
df.head()

,date,season,referee,league_code,team,enemy_team,match_result,ftg,enemy_ftg,s,st,f,c,y,r,enemy_s,enemy_st,enemy_f,enemy_c,enemy_y,enemy_r,ht_w,ht_d,ht_l,match_result_diff1,year,month,day,dow,weekofyear,is_weekend,match_result_lag_1,match_result_lag_2,match_result_lag_3,match_result_lag_4,match_result_lag_5,ftg_lag_1,ftg_lag_2,ftg_lag_3,ftg_lag_4,...,ht_w_roll_mean_10,ht_w_roll_std_10,ht_w_roll_mean_15,ht_w_roll_std_15,ht_w_roll_mean_20,ht_w_roll_std_20,ht_d_roll_mean_1,ht_d_roll_mean_2,ht_d_roll_std_2,ht_d_roll_mean_3,ht_d_roll_std_3,ht_d_roll_mean_4,ht_d_roll_std_4,ht_d_roll_mean_5,ht_d_roll_std_5,ht_d_roll_mean_7,ht_d_roll_std_7,ht_d_roll_mean_10,ht_d_roll_std_10,ht_d_roll_mean_15,ht_d_roll_std_15,ht_d_roll_mean_20,ht_d_roll_std_20,ht_l_roll_mean_1,ht_l_roll_mean_2,ht_l_roll_std_2,ht_l_roll_mean_3,ht_l_roll_std_3,ht_l_roll_mean_4,ht_l_roll_std_4,ht_l_roll_mean_5,ht_l_roll_std_5,ht_l_roll_mean_7,ht_l_roll_std_7,ht_l_roll_mean_10,ht_l_roll_std_10,ht_l_roll_mean_15,ht_l_roll_std_15,ht_l_roll_mean_20,ht_l_roll_std_20
0,2017-08-05,2017-2018,M Salisbury,E2,AFC Wimbledon,Scunthorpe,1,1,1,15.0,4.0,9.0,7.0,1.0,0.0,12.0,6.0,15.0,1.0,0.0,0.0,0,0,1,NaN,2017,8,5,5,31,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2017-08-12,2017-2018,C Breakspear,E2,AFC Wimbledon,Shrewsbury,0,0,1,6.0,2.0,14.0,2.0,2.0,0.0,14.0,5.0,19.0,4.0,2.0,0.0,0,0,1,NaN,2017,8,12,5,32,1,1.0,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,...,0.0,NaN,0.0,NaN,0.0,NaN,0.0,0.0,NaN,0.000000,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,0.00,NaN,1.0,1.0,NaN,1.000000,NaN,1.00,NaN,1.00,NaN,1.00,NaN,1.00,NaN,1.00,NaN,1.00,NaN
2,2017-08-19,2017-2018,J Brooks,E2,AFC Wimbledon,Fleetwood Town,0,0,2,9.0,1.0,11.0,8.0,1.0,0.0,4.0,2.0,11.0,4.0,2.0,0.0,0,0,1,-1.0,2017,8,19,5,33,1,0.0,1.0,NaN,NaN,NaN,0.0,1.0,NaN,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0,1.0,1.0,0.000000,1.000000,0.00000,1.00,0.0,1.00,0.0,1.00,0.0,1.00,0.0,1.00,0.0,1.00,0.0
3,2017-08-26,2017-2018,G Ward,E2,AFC Wimbledon,Doncaster,2,2,0,6.0,3.0,2.0,3.0,1.0,0.0,6.0,2.0,1.0,4.0,3.0,0.0,0,1,0,0.0,2017,8,26,5,34,1,0.0,0.0,1.0,NaN,NaN,0.0,0.0,1.0,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.00000,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0,0.00,0.0,1.0,1.0,0.000000,1.000000,0.00000,1.00,0.0,1.00,0.0,1.00,0.0,1.00,0.0,1.00,0.0,1.00,0.0
4,2017-09-02,2017-2018,C Boyeson,E2,AFC Wimbledon,Blackpool,0,0,1,4.0,1.0,15.0,4.0,3.0,1.0,18.0,9.0,10.0,7.0,1.0,0.0,0,1,0,2.0,2017,9,2,5,35,1,2.0,0.0,0.0,1.0,NaN,2.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.5,0.707107,0.333333,0.57735,0.25,0.5,0.25,0.5,0.25,0.5,0.25,0.5,0.25,0.5,0.25,0.5,0.0,0.5,0.707107,0.666667,0.57735,0.75,0.5,0.75,0.5,0.75,0.5,0.75,0.5,0.75,0.5,0.75,0.5


In [4]:
df["season"]

0        2017-2018
1        2017-2018
2        2017-2018
3        2017-2018
4        2017-2018
           ...    
41835    2018-2019
41836    2018-2019
41837    2018-2019
41838    2018-2019
41839    2018-2019
Name: season, Length: 41840, dtype: object

In [5]:
df = df.sort_values(by=["date"])
df

,date,season,referee,league_code,team,enemy_team,match_result,ftg,enemy_ftg,s,st,f,c,y,r,enemy_s,enemy_st,enemy_f,enemy_c,enemy_y,enemy_r,ht_w,ht_d,ht_l,match_result_diff1,year,month,day,dow,weekofyear,is_weekend,match_result_lag_1,match_result_lag_2,match_result_lag_3,match_result_lag_4,match_result_lag_5,ftg_lag_1,ftg_lag_2,ftg_lag_3,ftg_lag_4,...,ht_w_roll_mean_10,ht_w_roll_std_10,ht_w_roll_mean_15,ht_w_roll_std_15,ht_w_roll_mean_20,ht_w_roll_std_20,ht_d_roll_mean_1,ht_d_roll_mean_2,ht_d_roll_std_2,ht_d_roll_mean_3,ht_d_roll_std_3,ht_d_roll_mean_4,ht_d_roll_std_4,ht_d_roll_mean_5,ht_d_roll_std_5,ht_d_roll_mean_7,ht_d_roll_std_7,ht_d_roll_mean_10,ht_d_roll_std_10,ht_d_roll_mean_15,ht_d_roll_std_15,ht_d_roll_mean_20,ht_d_roll_std_20,ht_l_roll_mean_1,ht_l_roll_mean_2,ht_l_roll_std_2,ht_l_roll_mean_3,ht_l_roll_std_3,ht_l_roll_mean_4,ht_l_roll_std_4,ht_l_roll_mean_5,ht_l_roll_std_5,ht_l_roll_mean_7,ht_l_roll_std_7,ht_l_roll_mean_10,ht_l_roll_std_10,ht_l_roll_mean_15,ht_l_roll_std_15,ht_l_roll_mean_20,ht_l_roll_std_20
3823,2017-07-28,2017-2018,Unknown,D2,Bochum,St Pauli,0,0,1,17.0,5.0,12.0,8.0,1.0,0.0,16.0,7.0,10.0,2.0,1.0,0.0,0,1,0,NaN,2017,7,28,4,30,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.111111,0.333333,0.214286,0.425815,0.210526,0.418854,NaN,0.0,NaN,0.000000,0.00000,0.333333,0.57735,0.50,0.577350,0.500000,0.547723,0.444444,0.527046,0.500000,0.518875,0.473684,0.512989,NaN,1.0,NaN,1.000000,0.000000,0.666667,0.57735,0.50,0.577350,0.500000,0.547723,0.444444,0.527046,0.285714,0.468807,0.315789,0.477567
35090,2017-07-28,2017-2018,Unknown,D2,St Pauli,Bochum,2,1,0,16.0,7.0,10.0,2.0,1.0,0.0,17.0,5.0,12.0,8.0,1.0,0.0,0,1,0,NaN,2017,7,28,4,30,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.111111,0.333333,0.071429,0.267261,0.105263,0.315302,NaN,1.0,NaN,1.000000,0.00000,0.666667,0.57735,0.75,0.500000,0.666667,0.516398,0.666667,0.500000,0.714286,0.468807,0.736842,0.452414,NaN,0.0,NaN,0.000000,0.000000,0.333333,0.57735,0.25,0.500000,0.166667,0.408248,0.222222,0.440959,0.214286,0.425815,0.157895,0.374634
11417,2017-07-29,2017-2018,Unknown,D2,Darmstadt,Greuther Furth,2,1,0,10.0,5.0,13.0,2.0,1.0,0.0,10.0,1.0,21.0,5.0,4.0,0.0,0,1,0,NaN,2017,7,29,5,30,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.222222,0.440959,0.285714,0.468807,0.210526,0.418854,NaN,0.0,NaN,0.000000,0.00000,0.333333,0.57735,0.25,0.500000,0.333333,0.516398,0.444444,0.527046,0.428571,0.513553,0.526316,0.512989,NaN,0.0,NaN,0.000000,0.000000,0.000000,0.00000,0.25,0.500000,0.333333,0.516398,0.333333,0.500000,0.285714,0.468807,0.263158,0.452414
2526,2017-07-29,2017-2018,Unknown,D2,Bielefeld,Regensburg,2,2,1,19.0,5.0,13.0,7.0,3.0,1.0,14.0,3.0,12.0,4.0,3.0,0.0,0,1,0,NaN,2017,7,29,5,30,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.444444,0.527046,0.428571,0.513553,0.526316,0.512989,NaN,0.0,NaN,0.000000,0.00000,0.000000,0.00000,0.00,0.000000,0.333333,0.516398,0.444444,0.527046,0.500000,0.518875,0.421053,0.507257,NaN,0.0,NaN,0.500000,0.707107,0.333333,0.57735,0.25,0.500000,0.166667,0.408248,0.111111,0.333333,0.071429,0.267261,0.052632,0.229416
38352,2017-07-29,2017-2018,Unknown,D2,Union Berlin,Ingolstadt,2,1,0,13.0,3.0,17.0,3.0,3.0,0.0,13.0,4.0,13.0,3.0,0.0,0.0,0,1,0,NaN,2017,7,29,5,30,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.333333,0.500000,0.285714,0.468807,0.368421,0.495595,NaN,0.0,NaN,0.000000,0.00000,0.333333,0.57735,0.25,0.500000,0.166667,0.408248,0.444444,0.527046,0.500000,0.518875,0.421053,0.507257,NaN,1.0,NaN,1.000000,0.000000,0.666667,0.57735,0.50,0.577350,0.333333,0.516398,0.222222,0.440959,0.214286,0.425815,0.210526,0.418854
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4770,2025-05-25,2024-2025,L Smith,E0,Bournemouth,Leicester,2,2,0,20.0,7.0,19.0,6.0,0.0,0.0,3.0,0.0,16.0,1.0,2.0,0.0,0,1,0,0.0,2025,

In [14]:
cat_columns = [
    "referee",
    "team",
    "enemy_team",
    "league_code"
]

In [15]:
unknown_cols = [
    "match_result", 
    "ftg", 
    "enemy_ftg", 
    "s",
    "st",
    "f",
    "c",
    "y",
    "r",
    "enemy_s",
    "enemy_st",
    "enemy_f",
    "enemy_c",
    "enemy_y",
    "enemy_r",
    "ht_w",
    "ht_d",
    "ht_l",
    "date",
    "season"
    ]

In [16]:
numeric_columns = list(set(df.columns) - set(cat_columns) - set(unknown_cols))
len(numeric_columns)

403

Одна модель на все команды (ряды)

In [17]:
val_season = "2024-2025"

In [18]:
df.head()

,date,season,referee,league_code,team,enemy_team,match_result,ftg,enemy_ftg,s,st,f,c,y,r,enemy_s,enemy_st,enemy_f,enemy_c,enemy_y,enemy_r,ht_w,ht_d,ht_l,match_result_diff1,year,month,day,dow,weekofyear,is_weekend,match_result_lag_1,match_result_lag_2,match_result_lag_3,match_result_lag_4,match_result_lag_5,ftg_lag_1,ftg_lag_2,ftg_lag_3,ftg_lag_4,...,ht_w_roll_mean_10,ht_w_roll_std_10,ht_w_roll_mean_15,ht_w_roll_std_15,ht_w_roll_mean_20,ht_w_roll_std_20,ht_d_roll_mean_1,ht_d_roll_mean_2,ht_d_roll_std_2,ht_d_roll_mean_3,ht_d_roll_std_3,ht_d_roll_mean_4,ht_d_roll_std_4,ht_d_roll_mean_5,ht_d_roll_std_5,ht_d_roll_mean_7,ht_d_roll_std_7,ht_d_roll_mean_10,ht_d_roll_std_10,ht_d_roll_mean_15,ht_d_roll_std_15,ht_d_roll_mean_20,ht_d_roll_std_20,ht_l_roll_mean_1,ht_l_roll_mean_2,ht_l_roll_std_2,ht_l_roll_mean_3,ht_l_roll_std_3,ht_l_roll_mean_4,ht_l_roll_std_4,ht_l_roll_mean_5,ht_l_roll_std_5,ht_l_roll_mean_7,ht_l_roll_std_7,ht_l_roll_mean_10,ht_l_roll_std_10,ht_l_roll_mean_15,ht_l_roll_std_15,ht_l_roll_mean_20,ht_l_roll_std_20
3823,2017-07-28,2017-2018,Unknown,D2,Bochum,St Pauli,0,0,1,17.0,5.0,12.0,8.0,1.0,0.0,16.0,7.0,10.0,2.0,1.0,0.0,0,1,0,NaN,2017,7,28,4,30,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.111111,0.333333,0.214286,0.425815,0.210526,0.418854,NaN,0.0,NaN,0.0,0.0,0.333333,0.57735,0.50,0.57735,0.500000,0.547723,0.444444,0.527046,0.500000,0.518875,0.473684,0.512989,NaN,1.0,NaN,1.0,0.000000,0.666667,0.57735,0.50,0.57735,0.500000,0.547723,0.444444,0.527046,0.285714,0.468807,0.315789,0.477567
35090,2017-07-28,2017-2018,Unknown,D2,St Pauli,Bochum,2,1,0,16.0,7.0,10.0,2.0,1.0,0.0,17.0,5.0,12.0,8.0,1.0,0.0,0,1,0,NaN,2017,7,28,4,30,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.111111,0.333333,0.071429,0.267261,0.105263,0.315302,NaN,1.0,NaN,1.0,0.0,0.666667,0.57735,0.75,0.50000,0.666667,0.516398,0.666667,0.500000,0.714286,0.468807,0.736842,0.452414,NaN,0.0,NaN,0.0,0.000000,0.333333,0.57735,0.25,0.50000,0.166667,0.408248,0.222222,0.440959,0.214286,0.425815,0.157895,0.374634
11417,2017-07-29,2017-2018,Unknown,D2,Darmstadt,Greuther Furth,2,1,0,10.0,5.0,13.0,2.0,1.0,0.0,10.0,1.0,21.0,5.0,4.0,0.0,0,1,0,NaN,2017,7,29,5,30,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.222222,0.440959,0.285714,0.468807,0.210526,0.418854,NaN,0.0,NaN,0.0,0.0,0.333333,0.57735,0.25,0.50000,0.333333,0.516398,0.444444,0.527046,0.428571,0.513553,0.526316,0.512989,NaN,0.0,NaN,0.0,0.000000,0.000000,0.00000,0.25,0.50000,0.333333,0.516398,0.333333,0.500000,0.285714,0.468807,0.263158,0.452414
2526,2017-07-29,2017-2018,Unknown,D2,Bielefeld,Regensburg,2,2,1,19.0,5.0,13.0,7.0,3.0,1.0,14.0,3.0,12.0,4.0,3.0,0.0,0,1,0,NaN,2017,7,29,5,30,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.444444,0.527046,0.428571,0.513553,0.526316,0.512989,NaN,0.0,NaN,0.0,0.0,0.000000,0.00000,0.00,0.00000,0.333333,0.516398,0.444444,0.527046,0.500000,0.518875,0.421053,0.507257,NaN,0.0,NaN,0.5,0.707107,0.333333,0.57735,0.25,0.50000,0.166667,0.408248,0.111111,0.333333,0.071429,0.267261,0.052632,0.229416
38352,2017-07-29,2017-2018,Unknown,D2,Union Berlin,Ingolstadt,2,1,0,13.0,3.0,17.0,3.0,3.0,0.0,13.0,4.0,13.0,3.0,0.0,0.0,0,1,0,NaN,2017,7,29,5,30,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.333333,0.500000,0.285714,0.468807,0.368421,0.495595,NaN,0.0,NaN,0.0,0.0,0.333333,0.57735,0.25,0.50000,0.166667,0.408248,0.444444,0.527046,0.500000,0.518875,0.421053,0.507257,NaN,1.0,NaN,1.0,0.000000,0.666667,0.57735,0.50,0.57735,0.333333,0.516398,0.222222,0.440959,0.214286,0.425815,0.210526,0.418854


In [19]:
kfold_steps = 5

In [20]:
acc_scores = []
for i in range(kfold_steps):
    kfold_step = i + 1

    val_df = df[df["season"] == val_season]
    fold_size = val_df.shape[0] // (kfold_steps + 2)
    val_size = min(200, fold_size)
    train_date_thr = ""
    val_date_thr = ""
    for date in sorted(val_df["date"].unique()):
        if train_date_thr != "" and val_df[val_df["date"] <= date].shape[0] > fold_size * kfold_step:
            break
        train_date_thr = date

    for date in sorted(val_df["date"].unique()):
        if val_date_thr != "" and val_df[(val_df["date"] <= date) & (val_df["date"] > train_date_thr)].shape[0] > val_size:
            break
        val_date_thr = date

    assert train_date_thr < val_date_thr

    X_train = df[~(
        (df["season"] == val_season)
        & (df["date"] > train_date_thr)
    )]
    
    X_val = df[(
        (df["season"] == val_season)
        & (df["date"] < val_date_thr) & (df["date"] > train_date_thr)
    )]
        
    X_test = df[(
        (df["season"] == val_season)
        & (df["date"] >= val_date_thr)
    )]
    
    y_train = X_train["match_result"]
    X_train = X_train.drop(columns=unknown_cols)

    y_val = X_val["match_result"]
    X_val = X_val.drop(columns=unknown_cols)
    
    y_test = X_test["match_result"]
    X_test = X_test.drop(columns=unknown_cols)
    
    print(X_train.shape, X_val.shape, X_test.shape)

    train_pool = Pool(
        data=X_train,
        label=y_train,
        cat_features=cat_columns
    )
    
    val_pool = Pool(
        data=X_val,
        label=y_val,
        cat_features=cat_columns
    )

    test_pool = Pool(
        data=X_test,
        label=y_test,
        cat_features=cat_columns
    )

    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.005,
        depth=6,
        loss_function='MultiClass',
        random_seed=42,
        verbose=100,
        cat_features=cat_columns,
    )
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        verbose=100
    )

    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    acc_score = accuracy_score(y_test, y_pred)
    acc_scores.append(acc_score)
    
    print("="*50)
    print(i, acc_score)
    print("="*50)
          
print(np.mean(acc_scores))

(37298, 407) (186, 407) (4356, 407)
0:	learn: 1.0983567	test: 1.0984215	best: 1.0984215 (0)	total: 113ms	remaining: 5m 38s
100:	learn: 1.0725733	test: 1.0801624	best: 1.0801624 (100)	total: 4.07s	remaining: 1m 56s
200:	learn: 1.0606014	test: 1.0729964	best: 1.0729964 (200)	total: 8.14s	remaining: 1m 53s
300:	learn: 1.0540332	test: 1.0701340	best: 1.0701340 (300)	total: 12.2s	remaining: 1m 49s
400:	learn: 1.0498269	test: 1.0686497	best: 1.0686387 (399)	total: 16.3s	remaining: 1m 45s
500:	learn: 1.0467268	test: 1.0678138	best: 1.0676403 (482)	total: 20.4s	remaining: 1m 41s
600:	learn: 1.0441257	test: 1.0670241	best: 1.0670193 (599)	total: 24.5s	remaining: 1m 37s
700:	learn: 1.0419205	test: 1.0665926	best: 1.0665601 (654)	total: 28.6s	remaining: 1m 33s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 1.06656013
bestIteration = 654

Shrink model to first 655 iterations.
0 0.453168044077135
(38052, 407) (188, 407) (3600, 407)
0:	learn: 1.0982940	test: 1.0986087	best: 1.0986

то же самое но обучаемся на последний сезон

In [70]:
acc_scores = []
for i in range(n - 10 - 1, n-1):
    X_train = df_with_tour[(df_with_tour["season"] == val_season) & (df_with_tour["tour"] < i)].drop(columns=unknown_cols)
    y_train = df_with_tour[(df_with_tour["season"] == val_season) & (df_with_tour["tour"] < i)]["match_result"]

    X_val = df_with_tour[(df_with_tour["season"] == val_season) & (df_with_tour["tour"] == i)].drop(columns=unknown_cols)
    y_val = df_with_tour[(df_with_tour["season"] == val_season) & (df_with_tour["tour"] == i)]["match_result"]
    
    X_test = df_with_tour[(df_with_tour["season"] == val_season) & (df_with_tour["tour"] == i + 1)].drop(columns=unknown_cols)
    y_test = df_with_tour[(df_with_tour["season"] == val_season) & (df_with_tour["tour"] == i + 1)]["match_result"]

    train_pool = Pool(
        data=X_train,
        label=y_train,
        cat_features=cat_columns
    )
    
    val_pool = Pool(
        data=X_val,
        label=y_val,
        cat_features=cat_columns
    )

    test_pool = Pool(
        data=X_test,
        label=y_test,
        cat_features=cat_columns
    )

    model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.005,
        depth=6,
        loss_function='MultiClass',
        random_seed=42,
        verbose=100,
        cat_features=cat_columns,
    )
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=50,
        verbose=100
    )

    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
    
    acc_score = accuracy_score(y_test, y_pred)
    acc_scores.append(acc_score)
    
    print("="*50)
    print(i, acc_score)
    print("="*50)
          
print(np.mean(acc_scores))

0:	learn: 1.0979129	test: 1.0978705	best: 1.0978705 (0)	total: 47.7ms	remaining: 2m 23s
100:	learn: 1.0293662	test: 1.0576783	best: 1.0576783 (100)	total: 4.31s	remaining: 2m 3s
200:	learn: 0.9761475	test: 1.0337912	best: 1.0337912 (200)	total: 8.55s	remaining: 1m 59s
300:	learn: 0.9315161	test: 1.0143083	best: 1.0143083 (300)	total: 12.9s	remaining: 1m 55s
400:	learn: 0.8927027	test: 1.0045730	best: 1.0045730 (400)	total: 17s	remaining: 1m 50s
500:	learn: 0.8586713	test: 0.9999159	best: 0.9999159 (500)	total: 21.1s	remaining: 1m 45s
600:	learn: 0.8271825	test: 0.9924402	best: 0.9924402 (600)	total: 25.5s	remaining: 1m 41s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9899244988
bestIteration = 641

Shrink model to first 642 iterations.
31 0.6
0:	learn: 1.0979687	test: 1.0981356	best: 1.0981356 (0)	total: 49.8ms	remaining: 2m 29s
100:	learn: 1.0306649	test: 1.0685476	best: 1.0685476 (100)	total: 4.15s	remaining: 1m 59s
200:	learn: 0.9790650	test: 1.0506415	best: 1